# The regime sets with every competitor: ARI, class-balanced measures, the six sets, a wider grid

**Computes:** (A) On the regime sets (the imbalance sweep at ratio 20 with 3, 5 and 8 classes, Ecoli, Dermatology, HCV data), every method of the paper at every setting of its grid with the partitions kept: FCM, T1d, IT2, FCMA, FCMA-d, csiFCM, csiFCM-d, MC-FCM, RKM-IT2FL and a Gaussian mixture with 16 settings; label-tuned comparisons of IT2 with each by ARI (claim rule and corrected test), and at the settings selected by ARI the class-balanced measures (balanced accuracy, macro-F1, classes found, recall of the smallest class), the share of the largest class kept in one cluster and the near-empty clusters. (B) csiFCM, csiFCM-d and the mixture on the six sets chosen in advance, against IT2, FCM and T1d from the records of 24. (C) IT2 on a wider grid, m_1 = 1.01 and m_2 ∈ {50, 100} added, on the regime sets: label-tuned values against FCM and T1d, the selected settings and the share of the grid above the best type-1 setting.

**Inputs:** checkpoints of 06, 07 (main sets), 10 (Dermatology), 24 (six sets); UCI data (Dermatology is downloaded here if the data folder of 10 is absent).
**Outputs:** `results/29/results_29.json`, checkpoints `runs29*_*.pkl` and a text report.


In [ ]:
import os, sys, time, json, pickle, platform, subprocess, glob, shutil, urllib.request
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
DATASETS = None
RUN_STAMP = time.strftime("%Y%m%d-%H%M%S")

RESULTS_DIR = os.environ.get("RESULTS_DIR", "results")
OUTDIR = os.path.join(RESULTS_DIR, "29")
OUTDIR = os.path.abspath(OUTDIR)
for sub in ("data", "reports"):
    os.makedirs(os.path.join(OUTDIR, sub), exist_ok=True)
import numpy as np, scipy, sklearn
print(f"colab {IN_COLAB} | python {platform.python_version()} | numpy {np.__version__} | scipy {scipy.__version__} | sklearn {sklearn.__version__}")
print(f"output dir: {OUTDIR}   run stamp: {RUN_STAMP}")
BASE = os.path.dirname(OUTDIR)
DIR_06 = os.environ.get("DIR_06", os.path.join(BASE, "06"))
DIR_07 = os.environ.get("DIR_07", os.path.join(BASE, "07"))
DIR_10 = os.environ.get("DIR_10", os.path.join(BASE, "10"))
DIR_12 = os.environ.get("DIR_12", os.path.join(BASE, "12"))
print(f"{len(glob.glob(os.path.join(OUTDIR, 'runs29*_*.pkl'))):3d} checkpoints of 29 in {OUTDIR}")
DIR_20 = os.environ.get("DIR_20", os.path.join(BASE, "20"))
DIR_22 = os.environ.get("DIR_22", os.path.join(BASE, "22"))
DIR_24 = os.environ.get("DIR_24", os.path.join(BASE, "24"))
DIR_15 = os.environ.get("DIR_15", os.path.join(BASE, "15"))

In [ ]:
import numpy as np
from itertools import permutations

EPS = 1e-12

def fcm_membership(D, m):
    p = 2.0 / (m - 1.0)
    z = -p * np.log(np.maximum(D, EPS))
    z -= z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def hr_bounds(D, m1, m2):
    C = D.shape[1]
    U1, U2 = fcm_membership(D, m1), fcm_membership(D, m2)
    s = 1.0 / (D[:, :, None] / np.maximum(D[:, None, :], EPS)).sum(axis=2)
    cond = s < 1.0 / C
    upper = np.where(cond, U1, U2)
    lower = np.where(cond, U2, U1)
    return lower, upper

def km_interval_1d(x, wl, wu):
    o = np.argsort(x, kind="stable"); xs = x[o]; l = wl[o]; u = wu[o]
    n = len(x)
    cl, cu = np.cumsum(l), np.cumsum(u)
    clx, cux = np.cumsum(l * xs), np.cumsum(u * xs)
    Tl, Tu, Tlx, Tux = cl[-1], cu[-1], clx[-1], cux[-1]
    k = np.arange(n + 1)
    Lk = np.concatenate([[0.0], cl]); Uk = np.concatenate([[0.0], cu])
    Lxk = np.concatenate([[0.0], clx]); Uxk = np.concatenate([[0.0], cux])
    vR_all = (Lxk + (Tux - Uxk)) / np.maximum(Lk + (Tu - Uk), EPS)
    vL_all = (Uxk + (Tlx - Lxk)) / np.maximum(Uk + (Tl - Lk), EPS)
    kR, kL = int(np.argmax(vR_all)), int(np.argmin(vL_all))
    selR = np.zeros(n, bool); selL = np.zeros(n, bool)
    rank = np.empty(n, int); rank[o] = np.arange(n)
    selR[:] = rank >= kR
    selL[:] = rank < kL
    return float(vL_all[kL]), float(vR_all[kR]), selL, selR

def it2fcm_update(X, lower, upper, mc):
    N, M = X.shape; C = lower.shape[1]
    V = np.zeros((C, M)); uL = np.zeros((N, C)); uR = np.zeros((N, C))
    wl, wu = lower ** mc, upper ** mc
    for j in range(C):
        accL = np.zeros(N); accR = np.zeros(N)
        for l in range(M):
            vL, vR, selL, selR = km_interval_1d(X[:, l], wl[:, j], wu[:, j])
            V[j, l] = 0.5 * (vL + vR)
            accL += np.where(selL, upper[:, j], lower[:, j])
            accR += np.where(selR, upper[:, j], lower[:, j])
        uL[:, j], uR[:, j] = accL / M, accR / M
    return V, uL, uR

def dist(X, V):
    return np.maximum(np.linalg.norm(X[:, None, :] - V[None, :, :], axis=2), EPS)

def run_fcm(X, V0, m, iters=300, tol=1e-7):
    V = V0.copy()
    for _ in range(iters):
        U = fcm_membership(dist(X, V), m); W = U ** m
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    U = fcm_membership(dist(X, V), m)
    return V, U.argmax(1)

def run_it2fcm(X, V0, m1, m2, mc=None, iters=300, tol=1e-7):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        Vn, uL, uR = it2fcm_update(X, lo, up, mc)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    _, uL, uR = it2fcm_update(X, lo, up, mc)
    hard_hr = (0.5 * (uL + uR)).argmax(1)
    hard_nearest = dist(X, V).argmin(1)
    return V, hard_hr, hard_nearest

def accuracy(labels, pred, C):
    best = 0.0
    for perm in permutations(range(C)):
        best = max(best, float(np.mean(np.array(perm)[pred] == labels)))
    return best

def centre_error(V, centres):
    C = len(centres); best = np.inf
    for perm in permutations(range(C)):
        best = min(best, float(np.mean(np.linalg.norm(V[list(perm)] - centres, axis=1))))
    return best

def init_pp(X, C, rng):
    idx = [rng.integers(len(X))]
    for _ in range(1, C):
        d2 = np.min(((X[:, None, :] - X[idx][None]) ** 2).sum(2), axis=1)
        idx.append(rng.choice(len(X), p=d2 / d2.sum()))
    return X[idx].copy()

def scenario(name, seed):
    r = np.random.default_rng(seed)
    if name == "equal":
        c = np.array([[0.0, 0.0], [4.0, 0.0]]); n = (100, 100); s = (1.0, 1.0)
        X = np.vstack([r.normal(c[k], s[k], size=(n[k], 2)) for k in range(2)])
    elif name == "volume":
        c = np.array([[0.0, 0.0], [5.0, 0.0]]); n = (100, 100); half = (2.5, 0.6)
        X = np.vstack([c[k] + r.uniform(-half[k], half[k], size=(n[k], 2)) for k in range(2)])
    elif name == "noise":
        c = np.array([[0.0, 0.0], [0.0, 4.0]]); n = (100, 100)
        X = np.vstack([r.normal(c[0], 1.0, size=(n[0], 2)), r.normal(c[1], 1.0, size=(n[1], 2)),
                       r.normal(c[0], 3.0, size=(40, 2))])
        y = np.r_[np.zeros(n[0]), np.ones(n[1]), np.zeros(40)].astype(int)
        return X, y, c, 2
    elif name == "density":
        c = np.array([[0.0, 0.0], [3.5, 0.0]]); n = (200, 30); s = (1.0, 1.0)
        X = np.vstack([r.normal(c[k], s[k], size=(n[k], 2)) for k in range(2)])
    elif name == "volume3":
        c = np.array([[0.0, 0.0], [5.0, 0.0], [2.5, 4.0]]); n = (100, 100, 100); s = (1.4, 0.4, 0.8)
        X = np.vstack([r.normal(c[k], s[k], size=(n[k], 2)) for k in range(3)])
    else:
        raise ValueError(name)
    y = np.concatenate([np.full(n[k], k) for k in range(len(n))])
    return X, y, c, len(n)

from scipy.optimize import linear_sum_assignment
def accuracy_h(labels, pred, C):
    M = np.zeros((C, C))
    np.add.at(M, (labels, pred), 1)
    r, c = linear_sum_assignment(-M)
    return float(M[r, c].sum() / len(labels))

def align(V, ref):
    Dm = np.linalg.norm(V[:, None] - ref[None], axis=2)
    r, c = linear_sum_assignment(Dm)
    out = np.zeros_like(ref); out[c] = V[r]
    return out

def centre_error_h(V, centres):
    return float(np.mean(np.linalg.norm(align(V, centres) - centres, axis=1)))

def real_dataset(name):
    from sklearn import datasets
    d = {"iris": datasets.load_iris, "wine": datasets.load_wine, "breast_cancer": datasets.load_breast_cancer}[name]()
    X = d.data.astype(float); X = (X - X.mean(0)) / X.std(0)
    y = d.target.astype(int)
    return X, y, len(np.unique(y))

def run_fcm_decoupled(X, V0, ma, mb, iters=300, tol=1e-7):
    V = V0.copy()
    for _ in range(iters):
        U = fcm_membership(dist(X, V), ma); W = U ** mb
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    return V, dist(X, V).argmin(1)

def it2fcm_update_fast(X, order, lower, upper, mc):
    N, M = X.shape; C = lower.shape[1]
    Xs = np.take_along_axis(X, order, axis=0)
    rank = np.empty_like(order); np.put_along_axis(rank, order, np.arange(N)[:, None].repeat(M, 1), axis=0)
    V = np.zeros((C, M)); uL = np.zeros((N, C)); uR = np.zeros((N, C))
    z = np.zeros((1, M))
    for j in range(C):
        wl = (lower[:, j] ** mc)[order]; wu = (upper[:, j] ** mc)[order]
        cl, cu = np.cumsum(wl, 0), np.cumsum(wu, 0)
        clx, cux = np.cumsum(wl * Xs, 0), np.cumsum(wu * Xs, 0)
        Lk, Uk = np.vstack([z, cl]), np.vstack([z, cu]); Lxk, Uxk = np.vstack([z, clx]), np.vstack([z, cux])
        Tl, Tu, Tlx, Tux = cl[-1], cu[-1], clx[-1], cux[-1]
        vR_all = (Lxk + (Tux - Uxk)) / np.maximum(Lk + (Tu - Uk), EPS)
        vL_all = (Uxk + (Tlx - Lxk)) / np.maximum(Uk + (Tl - Lk), EPS)
        kR, kL = vR_all.argmax(0), vL_all.argmin(0)
        vR = vR_all[kR, np.arange(M)]; vL = vL_all[kL, np.arange(M)]
        V[j] = 0.5 * (vL + vR)
        selR = rank >= kR[None, :]; selL = rank < kL[None, :]
        up, lo = upper[:, j][:, None], lower[:, j][:, None]
        uR[:, j] = np.where(selR, up, lo).mean(1); uL[:, j] = np.where(selL, up, lo).mean(1)
    return V, uL, uR

def run_it2fcm_fast(X, V0, m1, m2, mc=None, iters=300, tol=1e-7, order=None):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    order = np.argsort(X, axis=0, kind="stable") if order is None else order
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        Vn, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    _, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
    return V, (0.5 * (uL + uR)).argmax(1), dist(X, V).argmin(1)

def real_dataset2(name, data_dir="data"):
    import os, pandas as pd
    if name in ("iris", "wine", "wdbc"):
        return real_dataset({"wdbc": "breast_cancer"}.get(name, name))
    if name == "sonar":
        d = pd.read_csv(os.path.join(data_dir, "sonar.csv"), header=None)
        X = d.iloc[:, :-1].to_numpy(float); y = (d.iloc[:, -1] == "M").to_numpy(int)
    elif name == "pima":
        d = pd.read_csv(os.path.join(data_dir, "pima-indians-diabetes.csv"), header=None)
        X = d.iloc[:, :-1].to_numpy(float); y = d.iloc[:, -1].to_numpy(int)
    elif name == "wbc":
        d = pd.read_csv(os.path.join(data_dir, "breast-cancer-wisconsin.data"), header=None, na_values="?").dropna()
        X = d.iloc[:, 1:-1].to_numpy(float); y = (d.iloc[:, -1] == 4).to_numpy(int)
    else:
        raise ValueError(name)
    X = (X - X.mean(0)) / X.std(0)
    return X, y, len(np.unique(y))

EXTRA = {"seeds": "wheat-seeds.csv", "glass": "glass.csv", "ionosphere": "ionosphere.csv",
         "banknote": "banknote_authentication.csv", "ecoli": "ecoli.csv"}
def real_dataset3(name, data_dir="data"):
    import os, pandas as pd
    if name not in EXTRA:
        return real_dataset2(name, data_dir)
    d = pd.read_csv(os.path.join(data_dir, EXTRA[name]), header=None)
    X = d.iloc[:, :-1].to_numpy(float)
    y = pd.factorize(d.iloc[:, -1].astype(str).str.strip(), sort=True)[0]
    X = X[:, X.std(0) > 0]
    X = (X - X.mean(0)) / X.std(0)
    return X, y.astype(int), len(np.unique(y))


In [ ]:
import time
import numpy as np

def run_it2fcm_nt(X, V0, m1, m2, mc=None, iters=300, tol=1e-6):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        W = 0.5 * (lo ** mc + up ** mc)
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    return V, (0.5 * (lo + up)).argmax(1)

def contingency(y, p, C):
    M = np.zeros((int(y.max()) + 1, C)); np.add.at(M, (y, p), 1); return M

def ari_from_cont(M):
    comb = lambda x: x * (x - 1) / 2.0
    n = M.sum(); s_ij = comb(M).sum(); a = comb(M.sum(1)).sum(); b = comb(M.sum(0)).sum()
    exp = a * b / comb(n); mx = 0.5 * (a + b)
    return float((s_ij - exp) / (mx - exp)) if mx != exp else 1.0

def nmi_from_cont(M):
    n = M.sum(); P = M / n; pi = P.sum(1); pj = P.sum(0)
    nz = P > 0
    I = float((P[nz] * np.log(P[nz] / (pi[:, None] * pj[None, :])[nz])).sum())
    H = lambda q: float(-(q[q > 0] * np.log(q[q > 0])).sum())
    d = 0.5 * (H(pi) + H(pj))
    return I / d if d > 0 else 1.0

def acc_from_cont(M):
    from scipy.optimize import linear_sum_assignment
    r, c = linear_sum_assignment(-M); return float(M[r, c].sum() / M.sum())

def scores(y, p, C):
    M = contingency(y, p, C)
    return dict(acc=acc_from_cont(M), ari=ari_from_cont(M), nmi=nmi_from_cont(M))

def silhouette_pre(Dsub, lab):
    ks = np.unique(lab)
    if len(ks) < 2: return -1.0
    n = len(lab); a = np.zeros(n); b = np.full(n, np.inf)
    for k in ks:
        mk = lab == k; cnt = mk.sum()
        s = Dsub[:, mk].sum(1)
        own = lab == k
        a[own] = s[own] / max(cnt - 1, 1)
        other = ~own
        b[other] = np.minimum(b[other], s[other] / cnt)
    sing = np.array([(lab == lab[i]).sum() == 1 for i in range(n)])
    sil = np.where(sing, 0.0, (b - a) / np.maximum(np.maximum(a, b), EPS))
    return float(sil.mean())

def calinski(X, lab):
    ks = np.unique(lab); n, k = len(X), len(ks)
    if k < 2 or k >= n: return -1.0
    mu = X.mean(0); B = W = 0.0
    for c in ks:
        Xc = X[lab == c]; mc_ = Xc.mean(0)
        B += len(Xc) * ((mc_ - mu) ** 2).sum(); W += ((Xc - mc_) ** 2).sum()
    return float((B / (k - 1)) / (W / (n - k))) if W > 0 else -1.0

def noise_instance(frac, seed):
    r = np.random.default_rng(seed); c = np.array([[0.0, 0.0], [0.0, 4.0]]); k = int(round(frac * 100))
    X = np.vstack([r.normal(c[0], 1.0, (100, 2)), r.normal(c[1], 1.0, (100, 2)), r.normal(c[0], 3.0, (k, 2))])
    y = np.r_[np.zeros(100), np.ones(100), np.zeros(k)].astype(int)
    return X, y, 2

def imbalance_instance(C, ratio, seed):
    r = np.random.default_rng(seed)
    R = 3.5 / (2 * np.sin(np.pi / C)); ang = 2 * np.pi * np.arange(C) / C
    c = np.c_[R * np.cos(ang), R * np.sin(ang)]
    n = np.maximum(np.round(100 * ratio ** (-np.arange(C) / max(C - 1, 1))).astype(int), 3)
    X = np.vstack([r.normal(c[k], 1.0, (n[k], 2)) for k in range(C)])
    y = np.concatenate([np.full(n[k], k) for k in range(C)])
    return X, y, C


In [ ]:
from PIL import Image
GRID = (1.05, 1.1, 1.3, 1.5, 2.0, 3.0, 5.0, 10.0, 20.0)
PAIRS = [(a, b) for i, a in enumerate(GRID) for b in GRID[i + 1:]]
IT2_KEYS = [(a, b, mode) for (a, b) in PAIRS for mode in ("m1", "mid", "m2")]
DEC_KEYS = [(a, b) for a in GRID for b in GRID]
FAMILIES = {"FCM": list(GRID), "IT2FCM": IT2_KEYS, "IT2FCM-NT": IT2_KEYS, "T1dec": DEC_KEYS}
SYN = ("equal", "volume", "noise", "density", "volume3")
REAL = ("iris", "wine", "wdbc", "wbc", "sonar", "pima", "seeds", "glass", "ionosphere", "banknote", "ecoli")
NOISE = {f"noise{int(f*100):02d}": f for f in (0.0, 0.1, 0.2, 0.4, 0.6)}
IMB = {f"imb_C{C}_r{r}": (C, r) for C in (3, 5, 8) for r in (1, 5, 20)}
N_SYN, N_REAL, N_SWEEP, N_INIT, TOL, N_SIL = 40, 20, 20, 2, 1e-6, 1000

def get_instance(nm, s, data_dir):
    if nm in SYN:
        X, y, c, C = scenario(nm, 1000 + s); return X, y, C
    if nm in NOISE: return noise_instance(NOISE[nm], 6000 + s)
    if nm in IMB: return imbalance_instance(*IMB[nm], 7000 + s)
    X, y, C = real_dataset3(nm, data_dir); r = np.random.default_rng(3000 + s)
    idx = np.sort(np.concatenate([r.choice(np.where(y == k)[0], int(round(0.75 * (y == k).sum())), replace=False) for k in range(C)]))
    return X[idx], y[idx], C

def fit(fam, key, X, V0, order):
    if fam == "FCM": return run_fcm(X, V0, key, tol=TOL)
    if fam == "T1dec": return run_fcm_decoupled(X, V0, key[0], key[1], tol=TOL)
    a, b, mode = key; mc = {"m1": a, "mid": 0.5 * (a + b), "m2": b}[mode]
    if fam == "IT2FCM":
        V, hp, hn = run_it2fcm_fast(X, V0, a, b, mc=mc, tol=TOL, order=order); return V, hp
    return run_it2fcm_nt(X, V0, a, b, mc=mc, tol=TOL)

def run_instance06(X, y, C, s, evaluate=None):
    order = np.argsort(X, axis=0, kind="stable")
    r = np.random.default_rng(8000 + s); sub = np.sort(r.choice(len(X), min(N_SIL, len(X)), replace=False))
    Dsub = np.linalg.norm(X[sub][:, None] - X[sub][None], axis=2)
    rec = {f: {k: dict(ari=[], acc=[], nmi=[], f=[], sil=[], ch=[]) for k in keys} for f, keys in FAMILIES.items()}
    secs = {f: 0.0 for f in FAMILIES}
    for q in range(N_INIT):
        V0 = init_pp(X, C, np.random.default_rng(10_000 * (q + 1) + s))
        for fam, keys in FAMILIES.items():
            t0 = time.perf_counter()
            for k in keys:
                V, lab = fit(fam, k, X, V0, order)
                sc = evaluate(V, lab) if evaluate else scores(y, lab, C)
                d = rec[fam][k]
                for m in ("ari", "acc", "nmi"): d[m].append(sc[m])
                d["f"].append(sc.get("f", np.nan)); d["sil"].append(silhouette_pre(Dsub, lab[sub])); d["ch"].append(calinski(X, lab))
            secs[fam] += time.perf_counter() - t0
    return dict(rec=rec, sec_per_run={f: secs[f] / (N_INIT * len(FAMILIES[f])) for f in FAMILIES})

def dibco_instance(i, f, g):
    I8 = np.array(Image.open(f).convert("L")); G = np.array(Image.open(g).convert("L")) < 128
    hist_ink = np.bincount(I8[G].ravel(), minlength=256).astype(float)
    hist_bg = np.bincount(I8[~G].ravel(), minlength=256).astype(float)
    r = np.random.default_rng(5000 + i)
    X = (I8.ravel()[r.choice(I8.size, min(2000, I8.size), replace=False)] / 255.0)[:, None]
    def evaluate(V, lab):
        t = float(V.ravel().mean()); k = int(np.clip(np.ceil(t * 255.0), 0, 256))
        tp, fp = hist_ink[:k].sum(), hist_bg[:k].sum(); fn, tn = hist_ink.sum() - tp, hist_bg.sum() - fp
        M = np.array([[tn, fp], [fn, tp]])
        return dict(acc=float((tp + tn) / M.sum()), ari=ari_from_cont(M), nmi=nmi_from_cont(M),
                    f=float(2 * tp / max(2 * tp + fp + fn, 1)))
    out = run_instance06(X, np.zeros(len(X), int), 2, i, evaluate=evaluate)
    out["name"] = os.path.basename(f).replace("TRAIN_", "").replace("_source", "")
    return out


In [ ]:
def hr_bounds_minmax(D, m1, m2):
    U1, U2 = fcm_membership(D, m1), fcm_membership(D, m2)
    return np.minimum(U1, U2), np.maximum(U1, U2)

def run_it2fcm_mm(X, V0, m1, m2, mc=None, iters=300, tol=1e-6, order=None):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    order = np.argsort(X, axis=0, kind="stable") if order is None else order
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds_minmax(dist(X, V), m1, m2)
        Vn, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds_minmax(dist(X, V), m1, m2)
    _, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
    return V, (0.5 * (uL + uR)).argmax(1), dist(X, V).argmin(1)

MM = "IT2FCM-mm"

def run_instance07(X, y, C, s, evaluate=None):
    order = np.argsort(X, axis=0, kind="stable")
    r = np.random.default_rng(8000 + s); sub = np.sort(r.choice(len(X), min(N_SIL, len(X)), replace=False))
    Dsub = np.linalg.norm(X[sub][:, None] - X[sub][None], axis=2)
    rec = {k: dict(ari=[], acc=[], nmi=[], f=[], sil=[], ch=[]) for k in IT2_KEYS}
    t0 = time.perf_counter(); mism_pts = mism_runs = 0
    for q in range(N_INIT):
        V0 = init_pp(X, C, np.random.default_rng(10_000 * (q + 1) + s))
        for k in IT2_KEYS:
            a, b, mode = k; mc = {"m1": a, "mid": 0.5 * (a + b), "m2": b}[mode]
            V, hp, hn = run_it2fcm_mm(X, V0, a, b, mc=mc, tol=TOL, order=order)
            sc = evaluate(V, hp) if evaluate else scores(y, hp, C)
            d = rec[k]
            for m in ("ari", "acc", "nmi"): d[m].append(sc[m])
            d["f"].append(sc.get("f", np.nan)); d["sil"].append(silhouette_pre(Dsub, hp[sub])); d["ch"].append(calinski(X, hp))
            mm_ = int((hp != hn).sum()); mism_pts += mm_; mism_runs += int(mm_ > 0)
    return dict(rec={MM: rec}, sec_per_run={MM: (time.perf_counter() - t0) / (N_INIT * len(IT2_KEYS))},
                mism_points=mism_pts, mism_runs=mism_runs)

def dibco_instance07(i, f, g):
    I8 = np.array(Image.open(f).convert("L")); G = np.array(Image.open(g).convert("L")) < 128
    hist_ink = np.bincount(I8[G].ravel(), minlength=256).astype(float)
    hist_bg = np.bincount(I8[~G].ravel(), minlength=256).astype(float)
    r = np.random.default_rng(5000 + i)
    X = (I8.ravel()[r.choice(I8.size, min(2000, I8.size), replace=False)] / 255.0)[:, None]
    def evaluate(V, lab):
        t = float(V.ravel().mean()); k = int(np.clip(np.ceil(t * 255.0), 0, 256))
        tp, fp = hist_ink[:k].sum(), hist_bg[:k].sum(); fn, tn = hist_ink.sum() - tp, hist_bg.sum() - fp
        M = np.array([[tn, fp], [fn, tp]])
        return dict(acc=float((tp + tn) / M.sum()), ari=ari_from_cont(M), nmi=nmi_from_cont(M), f=float(2 * tp / max(2 * tp + fp + fn, 1)))
    out = run_instance07(X, np.zeros(len(X), int), 2, i, evaluate=evaluate)
    out["name"] = os.path.basename(f).replace("TRAIN_", "").replace("_source", "")
    return out


In [ ]:
from scipy import stats
MAIN = SYN + REAL + ("dibco",)
SWEEPS = tuple(NOISE) + tuple(IMB)
FAM5 = dict(FAMILIES); FAM5[MM] = IT2_KEYS
PAIRS5 = (("IT2FCM", "FCM"), ("IT2FCM", "T1dec"),
          (MM, "FCM"), (MM, "T1dec"), (MM, "IT2FCM"), (MM, "IT2FCM-NT"))

def halves(nm, R):
    ids = sorted(R); h = len(ids) // 2
    if nm != "dibco": return ids[:h], ids[h:]
    rng = np.random.default_rng(7); years = {i: R[i]["name"][5:9] for i in ids}; fold = {}
    for yv in sorted(set(years.values())):
        ix = [i for i in ids if years[i] == yv]; ix = [ix[j] for j in rng.permutation(len(ix))]
        for j, i in enumerate(ix): fold[i] = j % 2
    return [i for i in ids if fold[i] == 0], [i for i in ids if fold[i] == 1]

def tp(d):
    d = np.asarray(d, float)
    if len(d) < 2 or np.allclose(d, 0): return dict(mean=float(d.mean()) if len(d) else 0.0, t_p=1.0, w_p=1.0, n=len(d))
    t = float(stats.ttest_1samp(d, 0).pvalue)
    try: w = float(stats.wilcoxon(d).pvalue)
    except ValueError: w = 1.0
    return dict(mean=float(d.mean()), t_p=t if np.isfinite(t) else 1.0, w_p=w if np.isfinite(w) else 1.0, n=len(d))

def verdict(a, b):
    sig = lambda z: z["t_p"] < 0.05 and z["w_p"] < 0.05
    if sig(a) and sig(b) and a["mean"] > 0 and b["mean"] > 0: return "BETTER"
    if sig(a) and sig(b) and a["mean"] < 0 and b["mean"] < 0: return "WORSE"
    return "n.s."

def unsup_value(rec_f, metric, by):
    keys = list(rec_f); nq = len(rec_f[keys[0]][by]); vals = []
    for q in range(nq):
        k = max(keys, key=lambda kk: rec_f[kk][by][q]); vals.append(rec_f[k][metric][q])
    return float(np.mean(vals))

def merge(R06, R07):
    ids = sorted(set(R06) & set(R07)); R = {}
    for i in ids:
        R[i] = dict(R06[i]); R[i]["rec"] = dict(R06[i]["rec"]); R[i]["rec"][MM] = R07[i]["rec"][MM]
        R[i]["sec_per_run"] = dict(R06[i]["sec_per_run"]); R[i]["sec_per_run"][MM] = R07[i]["sec_per_run"][MM]
        R[i]["mism_points"] = R07[i].get("mism_points", 0); R[i]["mism_runs"] = R07[i].get("mism_runs", 0)
    return R

def analyse_dataset(nm, R, metric):
    A, B = halves(nm, R); out = {"n": len(R)}
    for by in ("sil", "ch"):
        val = {f: {i: unsup_value(R[i]["rec"][f], metric, by) for i in R} for f in FAM5}
        cmp = {}
        for (x, z) in PAIRS5:
            a = tp([val[x][i] - val[z][i] for i in A]); b = tp([val[x][i] - val[z][i] for i in B])
            cmp[f"{x} vs {z}"] = dict(A=a, B=b, verdict=verdict(a, b), mean=float(np.mean([val[x][i] - val[z][i] for i in R])))
        out[by] = dict(mean={f: float(np.mean(list(val[f].values()))) for f in FAM5}, cmp=cmp)
    lt = {}
    for sn, (tr, te) in (("A->B", (A, B)), ("B->A", (B, A))):
        sel = {f: max(FAM5[f], key=lambda k: np.mean([np.mean(R[i]["rec"][f][k][metric]) for i in tr])) for f in FAM5}
        test = {f: {i: float(np.mean(R[i]["rec"][f][sel[f]][metric])) for i in te} for f in FAM5}
        lt[sn] = dict(selected={f: str(sel[f]) for f in sel}, cmp={f"{x} vs {z}": tp([test[x][i] - test[z][i] for i in te]) for (x, z) in PAIRS5})
    out["label_tuned"] = dict(splits=lt, verdict={c: verdict(lt["A->B"]["cmp"][c], lt["B->A"]["cmp"][c]) for c in lt["A->B"]["cmp"]},
                              mean={c: float(np.mean([lt[s]["cmp"][c]["mean"] for s in lt])) for c in lt["A->B"]["cmp"]})
    out["sec_per_run"] = {f: float(np.median([R[i]["sec_per_run"][f] for i in R])) for f in FAM5}
    out["mm_partition_vs_nearest"] = dict(points=int(sum(R[i]["mism_points"] for i in R)), runs=int(sum(R[i]["mism_runs"] for i in R)))
    out["mm_fingerprint"] = float(np.mean([np.mean(R[i]["rec"][MM][k]["acc"]) for i in R for k in IT2_KEYS]))
    return out

def dataset_level(RES, names, key):
    rows = [n for n in names if n in RES]; res = {}
    for (x, z) in PAIRS5:
        c = f"{x} vs {z}"
        d = np.array([RES[n]["label_tuned"]["mean"][c] if key == "label_tuned" else RES[n][key]["cmp"][c]["mean"] for n in rows])
        w = float(stats.wilcoxon(d).pvalue) if np.any(np.abs(d) > 1e-12) else 1.0
        res[c] = dict(n=len(rows), median=float(np.median(d)), pos=int((d > 0).sum()), neg=int((d < 0).sum()), wilcoxon_p=w)
    if key != "label_tuned":
        M = np.array([[RES[n][key]["mean"][f] for f in FAM5] for n in rows])
        fr = stats.friedmanchisquare(*M.T); ranks = np.mean([stats.rankdata(-r) for r in M], axis=0)
        res["friedman"] = dict(p=float(fr.pvalue), mean_rank=dict(zip(FAM5, map(float, ranks))))
    return res


In [ ]:
MAXIT = 300
def fcm10(X, V0, m):
    V = V0.copy(); conv = False
    for it in range(1, MAXIT + 1):
        U = fcm_membership(dist(X, V), m); W = U ** m
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < TOL:
            V = Vn; conv = True; break
        V = Vn
    return V, fcm_membership(dist(X, V), m).argmax(1), None, it, conv

def t1d10(X, V0, ma, mb):
    V = V0.copy(); conv = False
    for it in range(1, MAXIT + 1):
        U = fcm_membership(dist(X, V), ma); W = U ** mb
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < TOL:
            V = Vn; conv = True; break
        V = Vn
    return V, dist(X, V).argmin(1), None, it, conv

def it2km10(X, V0, m1, m2, mc, order, bounds):
    V = V0.copy(); conv = False
    for it in range(1, MAXIT + 1):
        lo, up = bounds(dist(X, V), m1, m2)
        Vn, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
        if np.abs(Vn - V).max() < TOL:
            V = Vn; conv = True; break
        V = Vn
    lo, up = bounds(dist(X, V), m1, m2)
    _, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
    return V, (0.5 * (uL + uR)).argmax(1), dist(X, V).argmin(1), it, conv

def nt10(X, V0, m1, m2, mc):
    V = V0.copy(); conv = False
    for it in range(1, MAXIT + 1):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        W = 0.5 * (lo ** mc + up ** mc)
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < TOL:
            V = Vn; conv = True; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    return V, (0.5 * (lo + up)).argmax(1), None, it, conv

def fit10(fam, key, X, V0, order):
    if fam == "FCM": return fcm10(X, V0, key)
    if fam == "T1dec": return t1d10(X, V0, key[0], key[1])
    a, b, mode = key; mc = {"m1": a, "mid": 0.5 * (a + b), "m2": b}[mode]
    if fam == "IT2FCM": return it2km10(X, V0, a, b, mc, order, hr_bounds)
    if fam == MM: return it2km10(X, V0, a, b, mc, order, hr_bounds_minmax)
    return nt10(X, V0, a, b, mc)

def xie_beni(X, V, m=2.0):
    D = dist(X, V); U = fcm_membership(D, m)
    sep = min(float(((V[j] - V[k]) ** 2).sum()) for j in range(len(V)) for k in range(j + 1, len(V)))
    return float(((U ** m) * D ** 2).sum() / (len(X) * max(sep, EPS)))

def keys_for(fam, G):
    if fam == "FCM": return list(G)
    if fam == "T1dec": return [(a, b) for a in G for b in G]
    return [(a, b, mode) for i, a in enumerate(G) for b in G[i + 1:] for mode in ("m1", "mid", "m2")]

def run_instance10(X, y, C, s, fams, centres=None):
    order = np.argsort(X, axis=0, kind="stable")
    r = np.random.default_rng(8000 + s); sub = np.sort(r.choice(len(X), min(N_SIL, len(X)), replace=False))
    Dsub = np.linalg.norm(X[sub][:, None] - X[sub][None], axis=2)
    fields = ["ari", "acc", "nmi", "sil", "ch", "negxb", "it", "conv"] + (["negcerr"] if centres is not None else [])
    rec = {f: {k: {fl: [] for fl in fields + (["acc_n", "ari_n"] if f == MM else [])} for k in keys} for f, keys in fams.items()}
    V_store = {f: {k: [] for k in keys} for f, keys in fams.items()}; secs = {f: 0.0 for f in fams}
    for q in range(N_INIT):
        V0 = init_pp(X, C, np.random.default_rng(10_000 * (q + 1) + s))
        for fam, keys in fams.items():
            t0 = time.perf_counter()
            for k in keys:
                V, lab, near, it, conv = fit10(fam, k, X, V0, order)
                sc = scores(y, lab, C); d = rec[fam][k]
                for fl in ("ari", "acc", "nmi"): d[fl].append(sc[fl])
                d["sil"].append(silhouette_pre(Dsub, lab[sub])); d["ch"].append(calinski(X, lab)); d["negxb"].append(-xie_beni(X, V))
                d["it"].append(it); d["conv"].append(bool(conv))
                if centres is not None: d["negcerr"].append(-centre_error_h(V, centres))
                if fam == MM:
                    sn = scores(y, near, C); d["acc_n"].append(sn["acc"]); d["ari_n"].append(sn["ari"])
                V_store[fam][k].append(V.astype(np.float32))
            secs[fam] += time.perf_counter() - t0
    return dict(rec=rec, V=V_store, C=C, sec_per_run={f: secs[f] / (N_INIT * len(fams[f])) for f in fams})


In [ ]:
DATA_DIR = os.path.join(OUTDIR, "data"); os.makedirs(DATA_DIR, exist_ok=True)
MIRROR = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/"
UCI = "https://archive.ics.uci.edu/ml/machine-learning-databases/"
EXPECT = {"new_thyroid": (215, 5, 3), "dermatology": (358, 34, 6), "yeast": (1484, 8, 10), "page_blocks": (5473, 10, 5)}

def _get(url, fn):
    p = os.path.join(DATA_DIR, fn)
    if not os.path.exists(p) or os.path.getsize(p) == 0:
        urllib.request.urlretrieve(url, p)
    return p

def _parse(name, how, p):
    import pandas as pd
    if name == "new_thyroid" and how == "mirror":
        d = pd.read_csv(p, header=None); X, y = d.iloc[:, :5], d.iloc[:, 5]
    elif name == "new_thyroid":
        d = pd.read_csv(p, header=None); X, y = d.iloc[:, 1:], d.iloc[:, 0]
    elif name == "dermatology":
        d = pd.read_csv(p, header=None, na_values="?").dropna(); X, y = d.iloc[:, :34], d.iloc[:, 34]
    elif name == "yeast":
        d = pd.read_csv(p, header=None, sep=r"\s+"); X, y = d.iloc[:, 1:9], d.iloc[:, 9]
    elif name == "page_blocks":
        d = pd.read_csv(p, header=None, sep=r"\s+"); X, y = d.iloc[:, :10], d.iloc[:, 10]
    return X.to_numpy(float), y.astype(str).str.strip().to_numpy()

def _openml(name):
    import pandas as pd
    from sklearn.datasets import fetch_openml
    b = fetch_openml(name={"page_blocks": "page-blocks", "new_thyroid": "thyroid-new"}.get(name, name), version=1, as_frame=True, parser="auto")
    X = b.data.apply(pd.to_numeric, errors="coerce"); keep = ~X.isna().any(axis=1)
    return X[keep].to_numpy(float), b.target[keep].astype(str).str.strip().to_numpy()

def load_partb(name):
    cache = os.path.join(DATA_DIR, f"partb_{name}.npz")
    if os.path.exists(cache):
        z = np.load(cache, allow_pickle=False); return z["X"], z["y"], int(z["C"])
    tries = {"new_thyroid": [("mirror", MIRROR + "new-thyroid.csv", "new-thyroid.csv"), ("uci", UCI + "thyroid-disease/new-thyroid.data", "new-thyroid.data"), ("openml", None, None)],
             "dermatology": [("uci", UCI + "dermatology/dermatology.data", "dermatology.data"), ("openml", None, None)],
             "yeast": [("uci", UCI + "yeast/yeast.data", "yeast.data"), ("openml", None, None)],
             "page_blocks": [("uciZ", UCI + "page-blocks/page-blocks.data.Z", "page-blocks.data.Z"), ("openml", None, None)]}[name]
    errors = []
    for how, url, fn in tries:
        try:
            if how == "openml":
                Xr, yr = _openml(name)
            elif how == "uciZ":
                pz = _get(url, fn); p = pz[:-2]
                subprocess.run(f"gzip -dc '{pz}' > '{p}'", shell=True, check=True); Xr, yr = _parse(name, how, p)
            else:
                Xr, yr = _parse(name, how, _get(url, fn))
            break
        except Exception as e:
            errors.append(f"{how}: {e!r}"); Xr = None
    if Xr is None:
        raise RuntimeError(f"{name}: every source failed:\n  " + "\n  ".join(errors))
    import pandas as pd
    y = pd.factorize(yr, sort=True)[0].astype(int); C = len(np.unique(y))
    N0, M0, C0 = EXPECT[name]
    assert C == C0 and Xr.shape[1] == M0 and abs(len(Xr) - N0) <= 10, f"{name}: unexpected shape {Xr.shape}, C={C}"
    X = Xr[:, Xr.std(0) > 0]; X = (X - X.mean(0)) / X.std(0)
    np.savez(cache, X=X, y=y, C=C); return X, y, C


In [ ]:
for _f in ("sonar.csv", "pima-indians-diabetes.csv", "breast-cancer-wisconsin.data", "wheat-seeds.csv", "glass.csv",
           "ionosphere.csv", "banknote_authentication.csv", "ecoli.csv"):
    _get(MIRROR + _f, _f)
def fcma_membership(D, alpha, ma):
    z = np.log(np.maximum(alpha, 1e-300))[None, :] - (2.0 / (ma - 1.0)) * np.log(np.maximum(D, EPS))
    z -= z.max(axis=1, keepdims=True); e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)
def fcma10(X, V0, ma, mb, freeze_alpha=False, track=False):
    V = V0.copy(); C = len(V0); alpha = np.full(C, 1.0 / C); conv = False; J = []
    for it in range(1, MAXIT + 1):
        U = fcma_membership(dist(X, V), alpha, ma); W = U ** mb
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if not freeze_alpha:
            S = (W * dist(X, Vn) ** 2).sum(0); a = np.maximum(S, 1e-300) ** (1.0 / mb)
            alpha = np.maximum(a / a.sum(), 1e-12); alpha /= alpha.sum()
        if track:
            U2 = fcma_membership(dist(X, Vn), alpha, ma)
            J.append(float(((alpha ** (1 - ma))[None, :] * U2 ** ma * dist(X, Vn) ** 2).sum()))
        if np.abs(Vn - V).max() < TOL:
            V = Vn; conv = True; break
        V = Vn
    lab = fcma_membership(dist(X, V), alpha, ma).argmax(1)
    return (V, lab, alpha, it, conv, J) if track else (V, lab, alpha, it, conv)
def keys12(fam, G):
    return list(G) if fam == "FCMA" else [(a, b) for a in G for b in G]
def fit12(fam, key, X, V0):
    return fcma10(X, V0, key, key) if fam == "FCMA" else fcma10(X, V0, key[0], key[1])
chk = []
for nm in ("volume", "imb_C8_r20", "ecoli"):
    X, y, C = get_instance(nm, 0, DATA_DIR); V0 = init_pp(X, C, np.random.default_rng(10_000))
    for m in (1.5, 2.0, 5.0):
        Vf, labf, _, _, _ = fcm10(X, V0, m)
        Va, laba, _, _, _ = fcma10(X, V0, m, m, freeze_alpha=True)
        _, _, _, _, _, J = fcma10(X, V0, m, m, track=True)
        rises = sum(1 for a_, b_ in zip(J, J[1:]) if b_ > a_ * (1 + 1e-9) + 1e-12)
        chk.append(dict(set=nm, m=m, max_proto_diff_vs_fcm=float(np.abs(Va - Vf).max()), same_labels=bool((laba == labf).all()), objective_rises=rises, iters=len(J)))
for c in chk: print(c)
assert all(c["max_proto_diff_vs_fcm"] < 1e-8 and c["same_labels"] for c in chk), "FCMA with frozen size variables does not reproduce FCM"
LOG_CHECKS = chk
print("checks passed: frozen FCMA = FCM; objective increases:", sum(c["objective_rises"] for c in chk), "(should be 0)")


In [ ]:
G_B = (1.05, 1.5, 2.0, 3.0, 5.0, 10.0, 20.0)
PARTB_SETS = ("new_thyroid", "dermatology", "yeast", "page_blocks")
def grid_of(nm): return (G_B if nm in PARTB_SETS else GRID)
def n_inst(nm):
    return 14 if nm in PARTB_SETS else (N_SWEEP if (nm in IMB or nm in NOISE) else (N_SYN if nm in SYN else N_REAL))
def inst_any(nm, s):
    if nm in PARTB_SETS:
        z = np.load(os.path.join(DIR_10, "data", f"partb_{nm}.npz")); X, y, C = z["X"], z["y"], int(z["C"])
        r = np.random.default_rng(3000 + s); f = {"page_blocks": 0.3}.get(nm, 0.75)
        idx = np.sort(np.concatenate([r.choice(np.where(y == k)[0], max(2, int(round(f * (y == k).sum()))), replace=False) for k in range(C)]))
        return X[idx], y[idx], C
    return get_instance(nm, s, DATA_DIR)
def fit_any(fam, key, X, V0, order):
    if fam in ("FCMA", "FCMA-d"):
        V, lab, _, _, _ = fit12(fam, key, X, V0); return V, lab
    V, lab, _, _, _ = fit10(fam, key, X, V0, order); return V, lab
def keys_any(fam, G):
    return keys12(fam, G) if fam in ("FCMA", "FCMA-d") else keys_for(fam, G)
def sil_setup(X, s):
    r = np.random.default_rng(8000 + s); sub = np.sort(r.choice(len(X), min(N_SIL, len(X)), replace=False))
    return sub, np.linalg.norm(X[sub][:, None] - X[sub][None], axis=2)
def describe(X, y, lab, Cp):
    sz = np.bincount(lab, minlength=Cp); big = np.argmax(np.bincount(y))
    return int((sz <= max(2, 0.01 * len(X))).sum()), float(np.bincount(lab[y == big], minlength=Cp).max() / (y == big).sum())
def J2(X, V):
    D = dist(X, V); U = fcm_membership(D, 2.0); return float((U ** 2 * D ** 2).sum())
def ckpt(name):
    p = os.path.join(OUTDIR, name); return (pickle.load(open(p, "rb")) if os.path.exists(p) else {}), p
def save_ckpt(obj, p):
    with open(p + ".tmp", "wb") as fh: pickle.dump(obj, fh)
    os.replace(p + ".tmp", p)
def load_ref(nm):
    if nm in PARTB_SETS:
        R = pickle.load(open(os.path.join(DIR_10, f"runs10_{nm}.pkl"), "rb"))
    else:
        R = merge(pickle.load(open(os.path.join(DIR_06, f"runs06_{nm}.pkl"), "rb")), pickle.load(open(os.path.join(DIR_07, f"runs07_{nm}.pkl"), "rb")))
    p12 = os.path.join(DIR_12, f"runs12_{nm}.pkl"); R12 = pickle.load(open(p12, "rb")) if os.path.exists(p12) else {}
    return R, R12


In [ ]:
# The six further data sets: download, parse, check against the expected size, standardize; instances as in the paper.
import pandas as pd
SETS24 = ["zoo", "hcv", "wine_red", "steel", "ctg", "shuttle"]
if os.environ.get("SETS27"): SETS24 = os.environ["SETS27"].split(",")
if os.environ.get("SETS24"): SETS24 = os.environ["SETS24"].split(",")
NAMES24 = {"zoo": "Zoo", "hcv": "HCV data", "wine_red": "Wine Quality (red)", "steel": "Steel Plates Faults",
           "ctg": "Cardiotocography", "shuttle": "Shuttle"}
EXPECT24 = {"zoo": (101, 16, 7), "hcv": (589, 12, 5), "wine_red": (1599, 11, 6), "steel": (1941, 27, 7),
            "ctg": (2126, 21, 10), "shuttle": (58000, 9, 7)}
FRAC24 = {"shuttle": 0.05}
N_INST24 = 14
CTG_FEATS = ["LB", "AC", "FM", "UC", "DL", "DS", "DP", "ASTV", "MSTV", "ALTV", "MLTV", "Width", "Min", "Max", "Nmax",
             "Nzeros", "Mode", "Mean", "Median", "Variance", "Tendency"]

def _parse24(name):
    if name == "zoo":
        d = pd.read_csv(_get(UCI + "zoo/zoo.data", "zoo.data"), header=None)
        return d.iloc[:, 1:17], d.iloc[:, 17]
    if name == "hcv":
        d = pd.read_csv(_get(UCI + "00571/hcvdat0.csv", "hcvdat0.csv"))
        d = d.drop(columns=[d.columns[0]])
        d["Sex"] = (d["Sex"].astype(str).str.strip().str.lower() == "m").astype(float)
        feats = [c for c in d.columns if c != "Category"]
        d[feats] = d[feats].apply(pd.to_numeric, errors="coerce"); d = d.dropna()
        return d[feats], d["Category"]
    if name == "wine_red":
        d = pd.read_csv(_get(UCI + "wine-quality/winequality-red.csv", "winequality-red.csv"), sep=";")
        return d.iloc[:, :11], d.iloc[:, 11]
    if name == "steel":
        d = pd.read_csv(_get(UCI + "00198/Faults.NNA", "Faults.NNA"), sep=r"\s+", header=None)
        Y = d.iloc[:, 27:34].to_numpy(); assert (Y.sum(1) == 1).all(), "steel: fault columns are not one-hot"
        return d.iloc[:, :27], pd.Series(Y.argmax(1))
    if name == "ctg":
        errs = []
        for how in ("uci", "openml"):
            try:
                if how == "uci":
                    p = _get(UCI + "00193/CTG.xls", "CTG.xls")
                    try:
                        import xlrd
                    except ImportError:
                        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xlrd"], check=True)
                    raw = pd.read_excel(p, sheet_name="Raw Data", header=None)
                    hdr = next(i for i in range(len(raw)) if {"LB", "CLASS"} <= set(raw.iloc[i].astype(str).str.strip()))
                    d = raw.iloc[hdr + 1:].copy(); d.columns = raw.iloc[hdr].astype(str).str.strip()
                else:
                    from sklearn.datasets import fetch_openml
                    d = fetch_openml(name="cardiotocography", version=1, as_frame=True, parser="auto").frame
                cols = {str(c).strip().lower(): c for c in d.columns}
                missing = [f for f in CTG_FEATS + ["CLASS"] if f.lower() not in cols]
                if missing: raise ValueError(f"columns not found {missing}")
                d = d[[cols[f.lower()] for f in CTG_FEATS + ["CLASS"]]].apply(pd.to_numeric, errors="coerce").dropna()
                return d.iloc[:, :21], d.iloc[:, 21].astype(int)
            except Exception as e:
                errs.append(f"{how}: {e!r}")
        raise RuntimeError("ctg: " + "; ".join(errs))
    if name == "shuttle":
        pz = _get(UCI + "statlog/shuttle/shuttle.trn.Z", "shuttle.trn.Z"); p1 = pz[:-2]
        if not os.path.exists(p1): subprocess.run(f"gzip -dc '{pz}' > '{p1}'", shell=True, check=True)
        p2 = _get(UCI + "statlog/shuttle/shuttle.tst", "shuttle.tst")
        d = pd.concat([pd.read_csv(p1, sep=r"\s+", header=None), pd.read_csv(p2, sep=r"\s+", header=None)], ignore_index=True)
        return d.iloc[:, :9], d.iloc[:, 9]
    raise KeyError(name)

def load24(name):
    cache = os.path.join(DATA_DIR, f"set24_{name}.npz")
    if os.path.exists(cache):
        z = np.load(cache, allow_pickle=False); return z["X"], z["y"], int(z["C"])
    Xd, yd = _parse24(name)
    X = Xd.to_numpy(float); y = pd.factorize(yd.astype(str).str.strip(), sort=True)[0].astype(int); C = len(np.unique(y))
    N0, M0, C0 = EXPECT24[name]
    assert C == C0 and X.shape[1] == M0 and abs(len(X) - N0) <= 10, f"{name}: unexpected shape {X.shape}, C={C}"
    X = X[:, X.std(0) > 0]; X = (X - X.mean(0)) / X.std(0)
    np.savez(cache, X=X, y=y, C=C); return X, y, C

def inst24(nm, s):
    X, y, C = DATA24[nm]
    r = np.random.default_rng(3000 + s); f = FRAC24.get(nm, 0.75)
    idx = np.sort(np.concatenate([r.choice(np.where(y == k)[0], max(2, int(round(f * (y == k).sum()))), replace=False) for k in range(C)]))
    return X[idx], y[idx], C

DATA24, LOAD_ERR = {}, {}
for nm in SETS24:
    try:
        DATA24[nm] = load24(nm); X, y, C = DATA24[nm]; sz = np.bincount(y)
        print(f"{nm:9s} N={len(X):6d} M={X.shape[1]:3d} C={C:2d} class sizes {sorted(sz.tolist(), reverse=True)} ratio {sz.max()/sz.min():.1f}")
    except Exception as e:
        LOAD_ERR[nm] = repr(e); print(f"{nm:9s} NOT AVAILABLE: {e!r}")

In [ ]:
# Shared helpers: reference per-setting records, label-tuned selection on halves, class-balanced scores.
MAIN = list(SYN) + list(REAL); IMBS = list(IMB)
REAL_SETS = list(REAL) + list(PARTB_SETS) + SETS24
RHO = {nm: (0.3 / 0.7 if nm == "page_blocks" else 0.25 / 0.75) for nm in REAL_SETS}

def load_records(nm):
    """Per-instance, per-setting records {s: {'C':..., 'rec': {fam: {key: {'ari': [...], 'acc': [...], ...}}}}} and their source."""
    if nm in SETS24:
        p = os.path.join(DIR_24, f"runs24_{nm}.pkl")
        if os.path.exists(p): return pickle.load(open(p, "rb")), "24"
        return None, None
    try:
        R, _ = load_ref(nm); return R, "06/07" if nm not in PARTB_SETS else "10"
    except Exception:
        pass
    for d, tag in ((DIR_20, "20"), (DIR_22, "22")):
        p = os.path.join(d, f"{'db20' if tag == '20' else 'rob22'}_{nm}.pkl")
        if os.path.exists(p): return pickle.load(open(p, "rb")), tag
    return None, None

def inst_vals(R, f, field="ari"):
    ids = sorted(R); return {k: np.array([np.mean(R[s]["rec"][f][k][field]) for s in ids]) for k in R[ids[0]]["rec"][f]}

def tp_nb(d, rho):
    """One-sample t-test, Wilcoxon test and the corrected resampled t-test (variance x (1/J + rho))."""
    d = np.asarray(d, float); n = len(d)
    if n < 2 or np.allclose(d, 0): return dict(mean=float(d.mean()) if n else 0.0, t_p=1.0, w_p=1.0, nb_p=1.0, n=n)
    t = float(stats.ttest_1samp(d, 0).pvalue)
    try: w = float(stats.wilcoxon(d).pvalue)
    except ValueError: w = 1.0
    se = np.sqrt(d.var(ddof=1) * (1.0 / n + rho)); tnb = d.mean() / se if se > 0 else 0.0
    return dict(mean=float(d.mean()), t_p=t, w_p=w, nb_p=float(2 * stats.t.sf(abs(tnb), n - 1)), n=n)

def verdict_nb(a, b):
    if a["nb_p"] < 0.05 and b["nb_p"] < 0.05 and a["mean"] > 0 and b["mean"] > 0: return "BETTER"
    if a["nb_p"] < 0.05 and b["nb_p"] < 0.05 and a["mean"] < 0 and b["mean"] < 0: return "WORSE"
    return "n.s."

def select_halves(inst_x):
    """Settings selected on each half by mean value; returns (key selected on A, key selected on B) and the halves."""
    n = len(next(iter(inst_x.values()))); h = n // 2; A, B = np.arange(h), np.arange(h, n)
    return max(inst_x, key=lambda k: inst_x[k][A].mean()), max(inst_x, key=lambda k: inst_x[k][B].mean()), A, B

def lt_compare(inst, x, z, rho=0.0):
    """Label-tuned comparison x - z with the claim rule and the corrected test; scoring on the other half."""
    sxA, sxB, A, B = select_halves(inst[x]); szA, szB, _, _ = select_halves(inst[z])
    a = tp_nb(inst[x][sxA][B] - inst[z][szA][B], rho); b = tp_nb(inst[x][sxB][A] - inst[z][szB][A], rho)
    return dict(mean=float(np.mean([a["mean"], b["mean"]])), verdict=verdict(a, b), verdict_nb=verdict_nb(a, b),
                p=[a["t_p"], b["t_p"]], w=[a["w_p"], b["w_p"]], nb_p=[a["nb_p"], b["nb_p"]], selected=[[str(sxA), str(szA)], [str(sxB), str(szB)]])

def balanced_scores(y, lab, C):
    """Balanced accuracy, macro-F1 and recall per class after the optimal matching; classes found = recall >= 1/2."""
    from scipy.optimize import linear_sum_assignment
    Cp = max(C, lab.max() + 1); M = np.zeros((C, Cp))
    for a_, b_ in zip(y, lab): M[a_, b_] += 1
    r, c = linear_sum_assignment(-M); rec = np.zeros(C); prec = np.zeros(C)
    for i, j in zip(r, c):
        rec[i] = M[i, j] / max(M[i].sum(), 1); prec[i] = M[i, j] / max(M[:, j].sum(), 1)
    f1 = np.where(rec + prec > 0, 2 * rec * prec / np.maximum(rec + prec, 1e-12), 0.0)
    small = int(np.argmin(np.bincount(y, minlength=C)))
    return dict(bacc=float(rec.mean()), macro_f1=float(f1.mean()), found=int((rec >= 0.5).sum()), small_recall=float(rec[small]))

def instance_of(nm, s):
    if nm in SETS24: return inst24(nm, s)
    return inst_any(nm, s)

def report_line(d): return f"{100*d['mean']:+6.1f}{'*' if d['verdict']=='BETTER' else ('-' if d['verdict']=='WORSE' else ' ')}"
print("helpers ready; real sets with the correction:", len(REAL_SETS))

In [ ]:
# Methods not in the preamble (as in notebooks 18, 21, 26 and 27) and a common runner that keeps every partition.
from sklearn.mixture import GaussianMixture
import warnings

def csi_step(X, V, ma, mb, cond=True):
    C, N = len(V), len(X); U = fcm_membership(dist(X, V), ma)
    if cond:
        lab = U.argmax(1); P = np.bincount(lab, minlength=C) / N
        f = (1.0 - P[lab]) / (1.0 - P.min())
    else:
        f = np.ones(N)
    W = (f[:, None] * U) ** mb; s = W.sum(0); ok = s > EPS
    Vn = V.copy(); Vn[ok] = (W.T @ X)[ok] / s[ok][:, None]
    return Vn
def csifcm(X, V0, ma, mb, cond=True):
    V, _, _, it0, _ = t1d10(X, V0, ma, mb); conv = False
    for it in range(1, MAXIT + 1):
        Vn = csi_step(X, V, ma, mb, cond)
        if np.abs(Vn - V).max() < TOL:
            V = Vn; conv = True; break
        V = Vn
    return V, fcm_membership(dist(X, V), ma).argmax(1), it0 + it, conv

ALPHAS_MC = (0.5, 1.0, 2.0)
def mc_exponents(X, C, mL, mU, alpha):
    N = len(X); k = max(1, N // C)
    D = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=2); Ds = np.sort(D, axis=1)
    delta = Ds[:, :k].sum(1); lo, hi = delta.min(), delta.max()
    t = (delta - lo) / (hi - lo) if hi > lo else np.zeros(N)
    return mL + (mU - mL) * t ** alpha
def mcfcm_run(X, V0, m_i):
    V = V0.copy(); conv = False; p = (2.0 / (m_i - 1.0))[:, None]
    for it in range(1, MAXIT + 1):
        D = dist(X, V); z = -p * np.log(np.maximum(D, EPS)); z -= z.max(1, keepdims=True); U = np.exp(z); U /= U.sum(1, keepdims=True)
        W = U ** m_i[:, None]; Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < TOL:
            V = Vn; conv = True; break
        V = Vn
    D = dist(X, V); z = -p * np.log(np.maximum(D, EPS)); z -= z.max(1, keepdims=True); U = np.exp(z)
    return V, U.argmax(1), it, conv

ZETAS = (1.2, 1.4, 2.0); RKM_IMAX = 100
def rkm_it2fl(X, V0, m1, m2, zeta, imax=RKM_IMAX):
    N = len(X); C = len(V0); V = V0.copy(); prev = None; conv = False
    for it in range(1, imax + 1):
        D = dist(X, V); near = D.argmin(1); dmin = D[np.arange(N), near]
        B = D <= zeta * np.maximum(dmin, EPS)[:, None]
        boundary = B.sum(1) > 1
        beta = np.zeros((N, C)); beta[~boundary, near[~boundary]] = 1.0
        upper_size = np.array([(B[:, i]).sum() for i in range(C)], float)
        if boundary.any():
            idx = np.where(boundary)[0]
            for n in idx:
                Bn = np.where(B[n])[0]; d = np.maximum(D[n, Bn], EPS); k = len(Bn)
                s = 1.0 / (d[:, None] / d[None, :]).sum(1)
                u1 = 1.0 / ((d[:, None] / d[None, :]) ** (2.0 / (m1 - 1.0))).sum(1)
                u2 = 1.0 / ((d[:, None] / d[None, :]) ** (2.0 / (m2 - 1.0))).sum(1)
                low = np.where(s >= 1.0 / k, u1, u2); up = np.where(s < 1.0 / k, u1, u2)
                w = upper_size[Bn] / upper_size[Bn].sum()
                beta[n, Bn] = low + w * (up - low)
        lab = near.copy()
        if boundary.any():
            bb = beta[boundary]; best = bb.argmax(1); tie = (bb == bb.max(1, keepdims=True)).sum(1) > 1
            lab[boundary] = np.where(tie, near[boundary], best)
        assign = (lab, B)
        if prev is not None and np.array_equal(prev[0], lab) and np.array_equal(prev[1], B):
            conv = True; break
        prev = assign
        num = beta.T @ X; den = beta.sum(0)
        V = np.where(den[:, None] > 0, num / np.maximum(den[:, None], EPS), V)
    return V, lab, float(boundary.mean()), it, conv

GMM16 = [(ct, reg) for ct in ("spherical", "diag", "tied", "full") for reg in (1e-6, 1e-4, 1e-2, 1e-1)]
def gmm_fit(X, V0, ct, reg, seed):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        g = GaussianMixture(n_components=len(V0), covariance_type=ct, means_init=V0, random_state=seed, max_iter=300, tol=1e-6, reg_covar=reg)
        g.fit(X); return g.predict(X), bool(g.converged_)

def keys29(fam, G):
    if fam in ("FCM", "T1dec", MM): return keys_for(fam, G)
    if fam in ("FCMA", "FCMA-d"): return keys12(fam, G)
    if fam == "csiFCM": return list(G)
    if fam == "csiFCM-d": return [(a, b) for a in G for b in G]
    if fam == "MCFCM": return [(a, b, al) for i, a in enumerate(G) for b in G[i + 1:] for al in ALPHAS_MC]
    if fam == "RKM": return [(a, b, z) for i, a in enumerate(G) for b in G[i + 1:] for z in ZETAS]
    if fam == "GMM": return list(GMM16)
    raise KeyError(fam)
def fit29(fam, key, X, V0, order, seed):
    if fam in ("FCM", "T1dec", MM):
        V, lab, _, _, _ = fit10(fam, key, X, V0, order); return lab
    if fam in ("FCMA", "FCMA-d"):
        V, lab, _, _, _ = fit12(fam, key, X, V0); return lab
    if fam == "csiFCM": return csifcm(X, V0, key, key)[1]
    if fam == "csiFCM-d": return csifcm(X, V0, key[0], key[1])[1]
    if fam == "MCFCM": return mcfcm_run(X, V0, mc_exponents(X, len(V0), key[0], key[1], key[2]))[1]
    if fam == "RKM": return rkm_it2fl(X, V0, key[0], key[1], key[2])[1]
    if fam == "GMM": return gmm_fit(X, V0, key[0], key[1], seed)[0]
    raise KeyError(fam)

def instance29(nm, s):
    if nm in SETS24: return inst24(nm, s)
    if nm in PARTB_SETS and not os.path.exists(os.path.join(DIR_10, "data", f"partb_{nm}.npz")):
        X, y, C = load_partb(nm)
        r = np.random.default_rng(3000 + s); f = {"page_blocks": 0.3}.get(nm, 0.75)
        idx = np.sort(np.concatenate([r.choice(np.where(y == k)[0], max(2, int(round(f * (y == k).sum()))), replace=False) for k in range(C)]))
        return X[idx], y[idx], C
    return inst_any(nm, s)
def n_inst29(nm): return N_INST24 if nm in SETS24 else n_inst(nm)
def grid29(nm): return G_B if (nm in PARTB_SETS or nm in SETS24) else GRID

def run_family(nm, fam, tag, keys=None):
    """Every setting of a family on every instance of a set, partitions kept; checkpoint runs29{tag}_{fam}_{nm}.pkl."""
    out, path = ckpt(f"runs29{tag}_{fam}_{nm}.pkl"); G = grid29(nm); keys = keys or keys29(fam, G); n = n_inst29(nm); t0 = time.time(); new = 0
    for s in range(n):
        if s in out: continue
        X, y, C = instance29(nm, s); order = np.argsort(X, axis=0, kind="stable"); sub, Dsub = sil_setup(X, s)
        rec = {k: dict(ari=[], acc=[], sil=[], lab=[]) for k in keys}
        for q in range(N_INIT):
            seed = 10_000 * (q + 1) + s; V0 = init_pp(X, C, np.random.default_rng(seed))
            for k in keys:
                lab = np.asarray(fit29(fam, k, X, V0, order, seed)); sc = scores(y, lab, C); d = rec[k]
                d["ari"].append(sc["ari"]); d["acc"].append(sc["acc"]); d["sil"].append(silhouette_pre(Dsub, lab[sub])); d["lab"].append(lab.astype(np.int16))
        out[s] = dict(C=C, N=len(X), y=y.astype(np.int16), rec={fam: rec}); save_ckpt(out, path); new += 1
    print(f"  {nm:12s} {fam:9s} {len(out):3d} instances ({new} new, {len(keys)} settings) in {time.time()-t0:.0f}s", flush=True)
    return out

def lt_select(R, fam, field="ari"):
    """Settings selected on each half by the mean of `field`; returns the selected key for the scoring half of every instance."""
    ids = sorted(R); h = len(ids) // 2; A, B = ids[:h], ids[h:]; keys = list(R[ids[0]]["rec"][fam])
    mean_on = lambda k, I: float(np.mean([np.mean(R[s]["rec"][fam][k][field]) for s in I]))
    kA = max(keys, key=lambda k: mean_on(k, A)); kB = max(keys, key=lambda k: mean_on(k, B))
    return {**{s: kA for s in B}, **{s: kB for s in A}}, A, B

def measures_at(R, fam, chosen):
    """Per-instance ARI and class-balanced measures (mean over initializations) at the chosen settings."""
    val = {}
    for s, k in chosen.items():
        d = R[s]["rec"][fam][k]; y = R[s]["y"].astype(int); C = R[s]["C"]; X_N = R[s]["N"]
        bs = [balanced_scores(y, lab.astype(int), C) for lab in d["lab"]]
        sz = [np.bincount(lab.astype(int), minlength=C) for lab in d["lab"]]; big = np.argmax(np.bincount(y))
        kept = [np.bincount(lab.astype(int)[y == big], minlength=C).max() / (y == big).sum() for lab in d["lab"]]
        tiny = [int((z <= max(2, 0.01 * X_N)).sum()) for z in sz]
        val[s] = dict(ari=float(np.mean(d["ari"])), acc=float(np.mean(d["acc"])), bacc=float(np.mean([b["bacc"] for b in bs])),
                      macro_f1=float(np.mean([b["macro_f1"] for b in bs])), found=float(np.mean([b["found"] for b in bs])),
                      small_recall=float(np.mean([b["small_recall"] for b in bs])), kept=float(np.mean(kept)), near_empty=float(np.mean(tiny)))
    return val

def compare_vals(vx, vz, A, B, field, rho):
    d = {s: vx[s][field] - vz[s][field] for s in vx}
    a = tp_nb([d[s] for s in B], rho); b = tp_nb([d[s] for s in A], rho)
    return dict(mean=float(np.mean(list(d.values()))), verdict=verdict(a, b), verdict_nb=verdict_nb(a, b), p=[a["t_p"], b["t_p"]], w=[a["w_p"], b["w_p"]], nb_p=[a["nb_p"], b["nb_p"]])
print("methods and runner ready")

In [ ]:
# (A) Regime sets, every competitor at every setting with partitions kept; ARI comparisons and class-balanced measures
# at the ARI-selected settings.
REGIME = ["imb_C3_r20", "imb_C5_r20", "imb_C8_r20", "ecoli", "dermatology", "hcv"]
if os.environ.get("SETS29A"): REGIME = os.environ["SETS29A"].split(",")
FAMS_A = ("FCM", "T1dec", MM, "FCMA", "FCMA-d", "csiFCM", "csiFCM-d", "MCFCM", "RKM", "GMM")
if os.environ.get("FAMS29A"): FAMS_A = tuple(os.environ["FAMS29A"].split(","))
MEAS = ("ari", "acc", "bacc", "macro_f1", "found", "small_recall", "kept", "near_empty")
RUNS_A = {}; RES_A = {}; t_all = time.time()
for nm in REGIME:
    try: instance29(nm, 0)
    except Exception as e: print(f"{nm}: data not available ({e!r}), skipped"); continue
    RUNS_A[nm] = {}
    for fam in FAMS_A: RUNS_A[nm][fam] = run_family(nm, fam, "a")
    rho = RHO.get(nm, 0.0); res = dict(n=n_inst29(nm), rho=rho, grid=len(grid29(nm)), real=nm in REAL_SETS, families={}, compare={})
    vals = {}
    for fam in FAMS_A:
        R = RUNS_A[nm][fam]; chosen, A, B = lt_select(R, fam, "ari"); v = measures_at(R, fam, chosen); vals[fam] = (v, A, B)
        res["families"][fam] = dict(settings=len(R[sorted(R)[0]]["rec"][fam]), selected=sorted({str(k) for k in chosen.values()}),
                                    level={m: float(np.mean([v[s][m] for s in v])) for m in MEAS})
    vI, A, B = vals[MM]
    for fam in FAMS_A:
        if fam == MM: continue
        vZ = vals[fam][0]
        res["compare"][f"IT2 vs {fam}"] = {m: compare_vals(vI, vZ, A, B, m, rho if m in ("ari", "acc") else rho) for m in ("ari", "acc", "bacc", "macro_f1", "small_recall")}
    RES_A[nm] = res
    line = " ".join(f"{fam}: {report_line(res['compare'][f'IT2 vs {fam}']['ari'])}/{100*res['compare'][f'IT2 vs {fam}']['bacc']['mean']:+.1f}{'*' if res['compare'][f'IT2 vs {fam}']['bacc']['verdict']=='BETTER' else ('-' if res['compare'][f'IT2 vs {fam}']['bacc']['verdict']=='WORSE' else ' ')}" for fam in FAMS_A if fam != MM)
    print(f"{nm:12s} IT2 minus (ARI / balanced acc.): {line}", flush=True)
print(f"(A) done in {time.time()-t_all:.0f}s")

In [ ]:
# (B) csiFCM, csiFCM-d and the Gaussian mixture on the six sets chosen in advance, against IT2, FCM and T1d from the
# records of 24.
SETS_B = [nm for nm in SETS24 if nm in DATA24]
if os.environ.get("SETS29B"): SETS_B = os.environ["SETS29B"].split(",")
FAMS_B = ("csiFCM", "csiFCM-d", "GMM")
RES_B = {}; t_all = time.time()
for nm in SETS_B:
    try: instance29(nm, 0)
    except Exception as e: print(f"{nm}: data not available ({e!r}), skipped"); continue
    R24, src = load_records(nm)
    if R24 is None: print(f"{nm}: records of 24 not found, skipped"); continue
    ids = sorted(R24); inst = {f: inst_vals(R24, f, "ari") for f in ("FCM", "T1dec", MM)}
    res = dict(source=src, n=len(ids), rho=RHO.get(nm, 0.0), compare={})
    for fam in FAMS_B:
        R = run_family(nm, fam, "b"); inst[fam] = inst_vals(R, fam, "ari")
        for z in ("FCM", "T1dec"):
            res["compare"][f"{fam} vs {z}"] = lt_compare(inst, fam, z, RHO.get(nm, 0.0))
        res["compare"][f"IT2 vs {fam}"] = lt_compare(inst, MM, fam, RHO.get(nm, 0.0))
    RES_B[nm] = res
    print(f"{nm:9s} " + "  ".join(f"IT2-{fam} {report_line(res['compare'][f'IT2 vs {fam}'])} [{res['compare'][f'IT2 vs {fam}']['verdict_nb']}]" for fam in FAMS_B), flush=True)
print(f"(B) done in {time.time()-t_all:.0f}s")

SETS_C = ["imb_C3_r20", "imb_C5_r20", "imb_C8_r20", "ecoli", "dermatology", "hcv"]
if os.environ.get("SETS29C"): SETS_C = os.environ["SETS29C"].split(",")
M1_EXTRA, M2_EXTRA = (1.01,), (50.0, 100.0)
def wide_keys(G):
    Gw = (M1_EXTRA + tuple(G) + M2_EXTRA)
    full = [(a, b, mode) for i, a in enumerate(Gw) for b in Gw[i + 1:] for mode in ("m1", "mid", "m2")]
    old = set(keys_for(MM, G)); return [k for k in full if k not in old], full
RES_C = {}; t_all = time.time()
for nm in SETS_C:
    try: instance29(nm, 0)
    except Exception as e: print(f"{nm}: data not available ({e!r}), skipped"); continue
    Rref, src = load_records(nm)
    if Rref is None: print(f"{nm}: reference records not found, skipped"); continue
    G = grid29(nm); new, full = wide_keys(G)
    R = run_family(nm, MM, "c", keys=new)
    ids = sorted(set(Rref) & set(R)); rho = RHO.get(nm, 0.0)
    inst = {f: {k: np.array([np.mean(Rref[s]["rec"][f][k]["ari"]) for s in ids]) for k in Rref[ids[0]]["rec"][f]} for f in ("FCM", "T1dec", MM)}
    inst["IT2-wide"] = dict(inst[MM]); inst["IT2-wide"].update({k: np.array([np.mean(R[s]["rec"][MM][k]["ari"]) for s in ids]) for k in new})
    inst["IT2-new"] = {k: inst["IT2-wide"][k] for k in new}
    res = dict(source=src, n=len(ids), settings_old=len(inst[MM]), settings_new=len(new), compare={})
    for x in (MM, "IT2-wide"):
        for z in ("FCM", "T1dec"): res["compare"][f"{x} vs {z}"] = lt_compare(inst, x, z, rho)
    res["compare"]["IT2-wide vs IT2"] = lt_compare(inst, "IT2-wide", MM, rho)
    bestF = max(inst["FCM"].values(), key=lambda v: v.mean()).mean(); bestT = max(inst["T1dec"].values(), key=lambda v: v.mean()).mean()
    grid_mean = lambda d: {k: float(v.mean()) for k, v in d.items()}
    gm_old, gm_new = grid_mean(inst[MM]), grid_mean(inst["IT2-new"])
    res["grid"] = dict(best_old=max(gm_old.values()), best_new=max(gm_new.values()), best_old_key=str(max(gm_old, key=gm_old.get)), best_new_key=str(max(gm_new, key=gm_new.get)),
                       share_above_fcm_old=float(np.mean([v > bestF for v in gm_old.values()])), share_above_fcm_new=float(np.mean([v > bestF for v in gm_new.values()])),
                       share_above_t1d_old=float(np.mean([v > bestT for v in gm_old.values()])), share_above_t1d_new=float(np.mean([v > bestT for v in gm_new.values()])),
                       by_m2={str(b): float(np.mean([gm_new[k] for k in gm_new if k[1] == b])) for b in M2_EXTRA}, m1_101={str(a): float(np.mean([gm_new[k] for k in gm_new if k[0] == a])) for a in M1_EXTRA})
    RES_C[nm] = res
    print(f"{nm:12s} IT2 vs FCM {report_line(res['compare'][MM + ' vs FCM'])} -> wide {report_line(res['compare']['IT2-wide vs FCM'])}; wide vs IT2 {report_line(res['compare']['IT2-wide vs IT2'])}; grid best {100*res['grid']['best_old']:.1f} -> {100*res['grid']['best_new']:.1f} at {res['grid']['best_new_key']}; selected {res['compare']['IT2-wide vs FCM']['selected']}", flush=True)
print(f"(C) done in {time.time()-t_all:.0f}s")

In [ ]:
# Report and result file.
LINES = []
def say(s=""): print(s); LINES.append(s)
def fmt(d): return report_line(d) + f" [{d['verdict_nb']}]"
say(f"NOTEBOOK 29  run {RUN_STAMP}")
say("(A) REGIME SETS: IT2 MINUS EACH COMPETITOR AT THE SETTINGS SELECTED BY ARI ON THE OTHER HALF (x100; * better / - worse by the claim rule; [corrected test])")
for nm, r in RES_A.items():
    say(f"  {nm}  ({r['n']} instances, grid {r['grid']}, rho {r['rho']:.2f})")
    say("    " + f"{'competitor':10s} {'ARI':>16s} {'accuracy':>16s} {'bal. acc.':>16s} {'macro-F1':>16s} {'small recall':>16s}")
    for k, c in r["compare"].items():
        say("    " + f"{k[7:]:10s} " + " ".join(f"{fmt(c[m]):>16s}" for m in ("ari", "acc", "bacc", "macro_f1", "small_recall")))
    say("    levels at the selected settings: " + "; ".join(f"{f}: ARI {100*v['level']['ari']:.1f} bacc {100*v['level']['bacc']:.1f} small {100*v['level']['small_recall']:.0f}% kept {100*v['level']['kept']:.0f}% near-empty {v['level']['near_empty']:.1f}" for f, v in r["families"].items()))
say("")
say("(B) SIX SETS CHOSEN IN ADVANCE: csiFCM, csiFCM-d AND THE MIXTURE (ARI x100)")
for nm, r in RES_B.items():
    say(f"  {nm:9s} " + "  ".join(f"{k}: {fmt(c)}" for k, c in r["compare"].items()))
say("")
say("(C) IT2 ON THE WIDER GRID (m1 = 1.01; m2 = 50, 100 added), ARI x100")
for nm, r in RES_C.items():
    g = r["grid"]
    say(f"  {nm:12s} paper grid: vs FCM {fmt(r['compare'][MM + ' vs FCM'])} vs T1d {fmt(r['compare'][MM + ' vs T1dec'])}; wide grid: vs FCM {fmt(r['compare']['IT2-wide vs FCM'])} vs T1d {fmt(r['compare']['IT2-wide vs T1dec'])}; wide minus paper grid {fmt(r['compare']['IT2-wide vs IT2'])}")
    say(f"               grid best {100*g['best_old']:.1f} ({g['best_old_key']}) -> added settings best {100*g['best_new']:.1f} ({g['best_new_key']}); share above FCM best {100*g['share_above_fcm_old']:.0f}% paper grid / {100*g['share_above_fcm_new']:.0f}% added settings; mean of added settings by m2 {g['by_m2']}; selected on the wide grid {r['compare']['IT2-wide vs FCM']['selected']}")
payload = dict(notebook="29", run_stamp=RUN_STAMP, regime=RES_A, six_sets=RES_B, wide_grid=RES_C)
for p in (os.path.join(OUTDIR, "results_29.json"), os.path.join(OUTDIR, "results_29_report.txt"), os.path.join(OUTDIR, "reports", f"results_29_report_{RUN_STAMP}.txt")):
    with open(p, "w") as fh:
        (json.dump(payload, fh, indent=1, default=str) if p.endswith(".json") else fh.write("\n".join(LINES) + "\n"))
print("saved", os.path.join(OUTDIR, "results_29.json"))